# 02 – Cleaning & Imputation

Dieses Notebook bereinigt den kombinierten Datensatz und behandelt fehlende Werte.

**Strategie je Variablentyp:**
- Boolesche Variablen → `False`
- Kategorische Unbekannte → `'Unbekannt'`
- Numerische Variablen → Median (gruppiert nach Gebäudetyp)
- Zeitreihen → Lineare Interpolation

In [5]:
import sys
import polars as pl
from pathlib import Path

current_dir = Path('..') / '03_src'
sys.path.append(str(current_dir))
root_dir = Path('..')

from config import (
    Schema, fill_false_list, fill_null_list,
    fill_median_list, fill_inter_list,
    fill_string_float, fill_string_boolean,
    fill_string_integer, fill_unbekannt_list
)
from data_cleaner import (
    fill_false, fill_null, fill_median, apply_interpolation,
    apply_string_to_datetime, apply_string_to_date,
    apply_string_to_float, apply_string_to_boolean, apply_string_to_integer,
    apply_unbekannt, compare_imputation, check_amount_nulls
)

## 2.1 Rohdaten laden

In [6]:
# Einlesen des im vorherigen Schritt gespeicherten Parquet-Datensatzes
data_combined = pl.read_parquet(
    root_dir / '02_data' / 'temp' / 'combined_data.parquet'
)

print(f'Zeilen: {len(data_combined)} | Spalten: {len(data_combined.columns)}')

Zeilen: 937456 | Spalten: 140


## 2.2 Imputation durchführen

In [7]:
data_combined_filled = (
    data_combined
    .pipe(apply_unbekannt, fill_unbekannt_list)   # Kategorisch: 'Unbekannt'
    .pipe(fill_false, fill_false_list)            # Boolean: False
    .pipe(fill_null, fill_null_list)              # Numerisch: 0
    .pipe(fill_median, fill_median_list, group_by_col='building_type')  # Median je Gebäudetyp
    .sort(['household_id', 'date'])
    .pipe(apply_interpolation, fill_inter_list)   # Zeitreihen: Lineare Interpolation
)

## 2.3 Imputations-Vergleich (Vor/Nach)

In [8]:
compare_imputation(
    df_before=data_combined,
    df_after=data_combined_filled,
    col_lists={
        'building_type':   ['building_type'],
        'fill_false_list': fill_false_list,
        'fill_null_list':  fill_null_list,
        'fill_median_list': fill_median_list,
        'fill_unbekannt':  fill_unbekannt_list,
        'fill_inter_list': fill_inter_list,
    }
)


-----------------------------------------------------------------
IMPUTATION VERGLEICH: data_combined vs. data_combined_filled
-----------------------------------------------------------------

  [building_type]  OK
   Nulls vorher  :  847,235
   Nulls nachher :        0  (847,235 behoben)

  [fill_false_list]  OK
   Nulls vorher  : 40,273,743
   Nulls nachher :        0  (40,273,743 behoben)

  [fill_null_list]  OK
   Nulls vorher  : 6,144,135
   Nulls nachher :        0  (6,144,135 behoben)

  [fill_median_list]  OK
   Nulls vorher  : 1,698,444
   Nulls nachher :        0  (1,698,444 behoben)

  [fill_unbekannt]  OK
   Nulls vorher  : 11,144,372
   Nulls nachher :        0  (11,144,372 behoben)

  [fill_inter_list]  OK
   Nulls vorher  :   47,492
   Nulls nachher :        0  (47,492 behoben)

-----------------------------------------------------------------
GESAMT  Nulls vorher: 60,155,421  |  Nulls nachher: 0  |  Behoben: 60,155,421 (100.0%)
----------------------------------------

liste,spalte,nulls_vorher,pct_vorher,nulls_nachher,pct_nachher,behoben,status
str,str,i64,f64,i64,f64,i64,str
"""building_type""","""building_type""",847235,90.38,0,0.0,847235,"""OK"""
"""fill_false_list""","""building_pvsystem_available""",847405,90.39,0,0.0,847405,"""OK"""
"""fill_false_list""","""installation_haspvsystem""",522088,55.69,0,0.0,522088,"""OK"""
"""fill_false_list""","""heatpump_installation_internet…",867637,92.55,0,0.0,867637,"""OK"""
"""fill_false_list""","""building_renovated_windows""",847235,90.38,0,0.0,847235,"""OK"""
…,…,…,…,…,…,…,…
"""fill_unbekannt""","""heatpump_installation_incorrec…",924482,98.62,0,0.0,924482,"""OK"""
"""fill_unbekannt""","""dhw_temperaturesetting_categor…",854197,91.12,0,0.0,854197,"""OK"""
"""fill_unbekannt""","""heatdistribution_expansiontank…",851640,90.85,0,0.0,851640,"""OK"""


## 2.4 Datentypkonvertierung

In [9]:
data_combined_cleaned = (
    data_combined_filled
    .pipe(apply_string_to_datetime, ['timestamp_local_right'], tz='Europe/Zurich')
    .pipe(apply_string_to_float, fill_string_float)
    .pipe(apply_string_to_boolean, fill_string_boolean)
    .pipe(apply_string_to_integer, fill_string_integer)
    .pipe(apply_string_to_date, ['visit_date'])
)

## 2.5 Zwischenspeichern als Parquet

In [10]:
output_path = root_dir / '02_data' / 'processed' / 'combined_data_cleaned.parquet'
output_path.parent.mkdir(parents=True, exist_ok=True)
data_combined_cleaned.write_parquet(str(output_path))
print(f'✅ Parquet gespeichert: {output_path.name}')

✅ Parquet gespeichert: combined_data_cleaned.parquet
